# Motion-S Starter: Text-to-Motion Retrieval Baseline

This notebook provides a simple and reliable baseline for the **Motion-S: Hierarchical Text-to-Motion Generation for Sign Language** competition.

The approach uses:
- Text preprocessing (gloss + sentence)
- TF-IDF feature extraction
- Nearest Neighbor retrieval
- Direct token transfer from the most similar training sample

The goal of this notebook is to:
- Produce a **valid submission**
- Provide a **fast and reproducible baseline**
- Serve as a starting point for further improvements

## Navigation

- [Setup and Data Loading](#setup-and-data-loading)
- [Token Utilities](#token-utilities)
- [Training Data Validation](#training-data-validation)
- [TF-IDF Feature Extraction](#tf-idf-feature-extraction)
- [Nearest Neighbor Retrieval](#nearest-neighbor-retrieval)
- [Token Prediction](#token-prediction)
- [Submission Validation](#submission-validation)
- [Save Submission](#save-submission)

## Setup and Data Loading

In [1]:
import os, re, gc, random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

INPUT_DIR = Path("/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language")

TRAIN_CSV  = INPUT_DIR / "train.csv"
TEST_CSV   = INPUT_DIR / "test.csv"
SAMPLE_SUB = INPUT_DIR / "sample_submission.csv"

OUT_PATH = Path("/kaggle/working/submission.csv")

train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB)

print("Train:", train.shape)
print("Test:", test.shape)

_ws_re = re.compile(r"\s+")

def norm_text(x):
    if not isinstance(x, str):
        return ""
    x = x.strip()
    x = _ws_re.sub(" ", x)
    return x

def build_text(df):
    g = df["gloss"].map(norm_text) if "gloss" in df.columns else ""
    s = df["sentence"].map(norm_text) if "sentence" in df.columns else ""
    return (g + " || " + g + " || " + s).fillna("")

train_text = build_text(train)
test_text  = build_text(test)

Train: (12467, 10)
Test: (3000, 3)


## Token Utilities

In [2]:
TOKEN_COLS = ["base_tokens", "residual_1", "residual_2", "residual_3", "residual_4", "residual_5"]

def parse_tokens(tok_str):
    if not isinstance(tok_str, str):
        return []
    tok_str = tok_str.strip()
    if tok_str == "":
        return []
    return [int(x) for x in tok_str.split()]

def tokens_to_str(tokens):
    return " ".join(map(str, tokens))

def enforce_len(tokens, min_len=40, max_len=800):
    if len(tokens) == 0:
        tokens = [random.randint(0, 511) for _ in range(min_len)]
        return tokens
    if len(tokens) < min_len:
        reps = (min_len + len(tokens) - 1) // len(tokens)
        tokens = (tokens * reps)[:min_len]
    if len(tokens) > max_len:
        tokens = tokens[:max_len]
    return tokens

## Training Data Validation

In [3]:
good_mask = np.ones(len(train), dtype=bool)
train_lens = []

for i, row in train.iterrows():
    lens = []
    ok = True
    for c in TOKEN_COLS:
        t = parse_tokens(row[c])
        if any((x < 0 or x > 511) for x in t):
            ok = False
            break
        lens.append(len(t))
    if (not ok) or (len(set(lens)) != 1) or (lens[0] < 1):
        good_mask[i] = False
    else:
        train_lens.append(lens[0])

train_good = train.loc[good_mask].reset_index(drop=True)
train_text_good = train_text.loc[good_mask].reset_index(drop=True)

print("Usable train rows:", len(train_good))

Usable train rows: 12463


## TF-IDF Feature Extraction

In [4]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    analyzer="char_wb",
    ngram_range=(3, 6),
    min_df=2,
    max_features=250000,
)

X_train = vectorizer.fit_transform(train_text_good)
X_test  = vectorizer.transform(test_text)

print("Feature shapes:", X_train.shape, X_test.shape)

Feature shapes: (12463, 42410) (3000, 42410)


## Nearest Neighbor Retrieval

In [5]:
nn = NearestNeighbors(n_neighbors=1, metric="cosine", algorithm="brute")
nn.fit(X_train)

dist, idx = nn.kneighbors(X_test, return_distance=True)

idx = idx.reshape(-1)
dist = dist.reshape(-1)

print("Retrieval completed")

Retrieval completed


## Token Prediction

In [6]:
pred = pd.DataFrame({"id": test["id"].values})

for c in TOKEN_COLS:
    pred[c] = ""

train_tokens_cache = {}
for c in TOKEN_COLS:
    train_tokens_cache[c] = [parse_tokens(x) for x in train_good[c].astype(str).tolist()]

MIN_LEN, MAX_LEN = 40, 800

for j in range(len(test)):
    k = int(idx[j])

    layers = []
    for c in TOKEN_COLS:
        layers.append(train_tokens_cache[c][k])

    L = min(len(t) for t in layers)
    if L <= 0:
        L = MIN_LEN

    base = enforce_len(layers[0][:L], MIN_LEN, MAX_LEN)
    L2 = len(base)

    fixed_layers = [base]
    for li in range(1, 6):
        t = layers[li][:L]
        t = enforce_len(t, L2, L2)
        fixed_layers.append(t)

    pred.at[j, "base_tokens"] = tokens_to_str(fixed_layers[0])
    pred.at[j, "residual_1"]  = tokens_to_str(fixed_layers[1])
    pred.at[j, "residual_2"]  = tokens_to_str(fixed_layers[2])
    pred.at[j, "residual_3"]  = tokens_to_str(fixed_layers[3])
    pred.at[j, "residual_4"]  = tokens_to_str(fixed_layers[4])
    pred.at[j, "residual_5"]  = tokens_to_str(fixed_layers[5])

## Submission Validation

In [7]:
def validate_row(r):
    lens = []
    for c in TOKEN_COLS:
        t = parse_tokens(r[c])
        if len(t) < 40 or len(t) > 800:
            return False
        if any((x < 0 or x > 511) for x in t):
            return False
        lens.append(len(t))
    return len(set(lens)) == 1

check_n = min(200, len(pred))
ok = 0
for i in range(check_n):
    ok += int(validate_row(pred.iloc[i]))

print("Validation:", ok, "/", check_n)

Validation: 200 / 200


## Save Submission

In [8]:
pred.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
display(pred.head())

Saved: /kaggle/working/submission.csv


,id,base_tokens,residual_1,residual_2,residual_3,residual_4,residual_5
0,6420249,111 42 42 42 42 42 42 42 42 42 42 42 42 193 19...,474 194 232 59 59 59 31 31 59 88 88 7 91 59 40...,432 375 58 454 454 157 211 321 202 202 73 169 ...,239 201 308 173 173 142 308 424 449 56 174 17 ...,318 500 22 353 71 423 251 71 108 20 71 20 229 ...,261 367 286 209 209 474 343 425 49 476 427 15 ...
1,6420682,130 276 174 174 174 174 174 174 174 50 50 50 2...,339 194 389 88 88 88 88 333 287 87 79 79 501 7...,406 406 452 202 321 321 321 321 325 356 75 75 ...,351 316 308 253 236 236 236 424 76 386 124 195...,347 125 500 95 274 424 424 498 499 498 156 177...,119 261 283 241 455 406 367 394 372 295 502 22...
2,6425789,130 272 272 272 272 272 272 2 47 47 47 47 186 ...,145 80 80 80 80 80 80 345 17 17 17 87 87 405 4...,454 129 300 300 300 140 185 27 48 157 157 310 ...,238 308 433 433 111 503 395 156 3 150 97 111 3...,64 500 54 54 502 426 204 442 424 114 390 425 1...,201 72 409 243 474 412 16 200 1 371 372 340 49...
3,6425858,130 276 174 174 145 145 145 174 174 50 50 50 2...,339 194 389 88 87 173 87 333 287 97 79 79 289 ...,406 48 286 286 453 346 6 321 325 354 348 246 4...,351 463 336 401 401 308 236 401 134 72 353 353...,64 32 353 456 409 222 290 74 320 498 156 318 2...,257 261 367 367 372 241 239 409 249 137 350 22...
4,6427530,130 276 174 174 174 145 145 174 174 50 50 50 2...,339 194 389 88 88 173 173 333 287 97 79 79 289...,406 48 452 286 321 346 346 321 325 354 348 75 ...,351 463 449 401 308 236 236 424 134 96 333 195...,64 32 234 456 111 361 361 498 320 17 498 318 2...,257 261 367 367 372 241 241 241 249 343 295 50...


## Summary

**What this notebook does**
- Uses a simple nearest-neighbor approach to generate motion tokens
- Runs fast and creates a valid submission without any model training

**What you can try next**
- Predict motion length more accurately
- Use several nearest neighbors instead of one
- Replace TF-IDF with better text embeddings